# Analyse NumPy — Data Science Salaries

## Objectif du notebook

Ce notebook illustre l'utilisation de **NumPy** pour les calculs scientifiques
dans le cadre du projet *Data Science Salary Analysis & Salary Prediction*.

Il couvre :

1. La création d'arrays avec `np.array()`.
2. Le slicing et la vectorisation.
3. Le broadcasting.
4. Les statistiques descriptives : moyenne, médiane, minimum, maximum,
   écart-type.
5. L'interprétation des résultats obtenus sur les salaires.

**Source des données :** `data/raw/DataScience_salaries_2025.csv`
(151 445 lignes, 11 colonnes, 0 valeur manquante).
Provenance et licence : voir `data/raw/README.md`.
Stratégie de nettoyage : voir `docs/STRATEGIE_NETTOYAGE.md`.

**Kernel requis :** `.venv` (Python 3.12.10, NumPy 2.5.4).

In [1]:
# Cellule 2 : imports et configuration
from pathlib import Path

import numpy as np
import pandas as pd

# Affichage lisible dans le notebook
np.set_printoptions(precision=2, suppress=True, linewidth=120)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Racine du projet et chemin du CSV
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "DataScience_salaries_2025.csv"

print("Racine du projet :", PROJECT_ROOT)
print("Fichier CSV :", CSV_PATH)
print("Le fichier existe :", CSV_PATH.exists())

Racine du projet : c:\Users\malak\Desktop\data-science-salary-analysis
Fichier CSV : c:\Users\malak\Desktop\data-science-salary-analysis\data\raw\DataScience_salaries_2025.csv
Le fichier existe : True


In [2]:
# Cellule 3 : vérification de l'environnement NumPy
print("Version de NumPy :", np.__version__)
print("Version de Pandas :", pd.__version__)

# Test rapide : création d'un array de contrôle
_a = np.array([1, 2, 3])
print("Array de contrôle :", _a)
print("Type :", type(_a))
print("dtype :", _a.dtype)

Version de NumPy : 2.5.4
Version de Pandas : 3.1.0
Array de contrôle : [1 2 3]
Type : <class 'numpy.ndarray'>
dtype : int64


## 1. Création d'arrays avec `np.array()`

### 1.1 Array 1D à partir d'une liste

On crée un array de salaires annuels en USD, à partir d'une liste Python.
On observe le `dtype` inféré par NumPy.

In [3]:
# Cellule 5 : array 1D de salaires
salaires = np.array([45000, 62000, 85000, 120000, 95000, 150000, 75000])

print("Array :", salaires)
print("Type :", type(salaires))
print("dtype :", salaires.dtype)
print("shape :", salaires.shape)
print("ndim :", salaires.ndim)
print("Taille (nombre d'éléments) :", salaires.size)
print("Taille mémoire (octets) :", salaires.nbytes)

Array : [ 45000  62000  85000 120000  95000 150000  75000]
Type : <class 'numpy.ndarray'>
dtype : int64
shape : (7,)
ndim : 1
Taille (nombre d'éléments) : 7
Taille mémoire (octets) : 56


In [4]:
# Cellule 6 : array 2D
# Chaque ligne = un niveau d'expérience, chaque colonne = une année
# (données illustratives basées sur l'ordre de grandeur observé à l'étape 10)
salaires_par_niveau_annee = np.array([
    [45000, 52000, 58000, 62000],   # EN (entry)
    [70000, 78000, 85000, 92000],   # MI (mid)
    [110000, 125000, 140000, 155000],  # SE (senior)
    [180000, 200000, 220000, 240000],  # EX (executive)
])

print("Array 2D :\n", salaires_par_niveau_annee)
print("shape :", salaires_par_niveau_annee.shape)
print("ndim :", salaires_par_niveau_annee.ndim)
print("Taille :", salaires_par_niveau_annee.size)
print("Taille mémoire (octets) :", salaires_par_niveau_annee.nbytes)

Array 2D :
 [[ 45000  52000  58000  62000]
 [ 70000  78000  85000  92000]
 [110000 125000 140000 155000]
 [180000 200000 220000 240000]]
shape : (4, 4)
ndim : 2
Taille : 16
Taille mémoire (octets) : 128


In [5]:
# Cellule 7 : forcer le dtype et comparer liste vs array
# Sans dtype explicite, NumPy infère int64
salaires_int = np.array([45000, 62000, 85000])
print("dtype inféré :", salaires_int.dtype)

# Avec dtype explicite (float64, utile pour les calculs statistiques)
salaires_float = np.array([45000, 62000, 85000], dtype=np.float64)
print("dtype explicite :", salaires_float.dtype)

# Comparaison liste Python vs array NumPy
liste_python = [45000, 62000, 85000, 120000, 95000]
array_numpy = np.array(liste_python)

print("\nListe Python :", liste_python, "| type :", type(liste_python))
print("Array NumPy   :", array_numpy, "| type :", type(array_numpy))

# Opération : multiplier chaque salaire par 1.1 (augmentation de 10 %)
# Sur une liste, il faudrait une boucle ou une compréhension.
# Sur un array, l'opération est vectorisée.
print("\n=== Multiplication par 1.1 ===")
print("Array NumPy (vectorisé) :", array_numpy * 1.1)

dtype inféré : int64
dtype explicite : float64

Liste Python : [45000, 62000, 85000, 120000, 95000] | type : <class 'list'>
Array NumPy   : [ 45000  62000  85000 120000  95000] | type : <class 'numpy.ndarray'>

=== Multiplication par 1.1 ===
Array NumPy (vectorisé) : [ 49500.  68200.  93500. 132000. 104500.]


## 2. Slicing et vectorisation

### 2.1 Slicing sur un array 1D

Le slicing NumPy suit la syntaxe `array[début:fin:pas]` (fin exclue).
Les indices négatifs comptent depuis la fin.

In [6]:
# Cellule 8 : slicing sur l'array 1D de salaires
print("Array complet :", salaires)
print()

print("3 premiers :", salaires[:3])
print("3 derniers :", salaires[-3:])
print("Du 2e au 5e (inclus) :", salaires[1:5])
print("Un sur deux :", salaires[::2])
print("Inversé :", salaires[::-1])
print()

# Sous-ensemble : salaires supérieurs à 100 000
masque = salaires > 100000
print("Masque booléen :", masque)
print("Salaires > 100k :", salaires[masque])
print("Nombre de salaires > 100k :", masque.sum())

Array complet : [ 45000  62000  85000 120000  95000 150000  75000]

3 premiers : [45000 62000 85000]
3 derniers : [ 95000 150000  75000]
Du 2e au 5e (inclus) : [ 62000  85000 120000  95000]
Un sur deux : [45000 85000 95000 75000]
Inversé : [ 75000 150000  95000 120000  85000  62000  45000]

Masque booléen : [False False False  True False  True False]
Salaires > 100k : [120000 150000]
Nombre de salaires > 100k : 2


In [7]:
# Cellule 9 : slicing sur l'array 2D
print("Matrice complète :\n", salaires_par_niveau_annee)
print()

print("Ligne 0 (EN) :", salaires_par_niveau_annee[0])
print("Ligne 2 (SE) :", salaires_par_niveau_annee[2])
print()

print("Colonne 0 (année 1) :", salaires_par_niveau_annee[:, 0])
print("Colonne 3 (année 4) :", salaires_par_niveau_annee[:, 3])
print()

print("Sous-matrice lignes 1-2, colonnes 1-2 :\n",
      salaires_par_niveau_annee[1:3, 1:3])
print()

# Salaire d'un niveau donné pour une année donnée
print("Salaire SE en année 3 (index 2) :", salaires_par_niveau_annee[2, 2])

Matrice complète :
 [[ 45000  52000  58000  62000]
 [ 70000  78000  85000  92000]
 [110000 125000 140000 155000]
 [180000 200000 220000 240000]]

Ligne 0 (EN) : [45000 52000 58000 62000]
Ligne 2 (SE) : [110000 125000 140000 155000]

Colonne 0 (année 1) : [ 45000  70000 110000 180000]
Colonne 3 (année 4) : [ 62000  92000 155000 240000]

Sous-matrice lignes 1-2, colonnes 1-2 :
 [[ 78000  85000]
 [125000 140000]]

Salaire SE en année 3 (index 2) : 140000


In [8]:
# Cellule 10 : vectorisation
print("=== Opérations vectorisées sur l'array 1D ===")
print("Salaires bruts :", salaires)
print("+10 % (vectorisé) :", (salaires * 1.10).round(0))
print("+5 000 (vectorisé) :", salaires + 5000)
print("Racine carrée :", np.sqrt(salaires).round(0))
print()

# Comparaison boucle vs vectorisation
import time

N = 1_000_000
grand_array = np.random.randint(30000, 200000, size=N)

# Version vectorisée
t0 = time.perf_counter()
resultat_vectorise = grand_array * 1.10
t1 = time.perf_counter()
temps_vectorise = t1 - t0

# Version boucle Python
t0 = time.perf_counter()
resultat_boucle = np.empty(N)
for i in range(N):
    resultat_boucle[i] = grand_array[i] * 1.10
t1 = time.perf_counter()
temps_boucle = t1 - t0

print(f"Temps vectorisé : {temps_vectorise*1000:.2f} ms")
print(f"Temps boucle    : {temps_boucle*1000:.2f} ms")
print(f"Ratio           : x{temps_boucle/temps_vectorise:.1f} plus rapide")

# Vérification que les deux versions donnent le même résultat
print("Résultats identiques :", np.allclose(resultat_vectorise, resultat_boucle))

=== Opérations vectorisées sur l'array 1D ===
Salaires bruts : [ 45000  62000  85000 120000  95000 150000  75000]
+10 % (vectorisé) : [ 49500.  68200.  93500. 132000. 104500. 165000.  82500.]
+5 000 (vectorisé) : [ 50000  67000  90000 125000 100000 155000  80000]
Racine carrée : [212. 249. 292. 346. 308. 387. 274.]

Temps vectorisé : 3.67 ms
Temps boucle    : 1918.37 ms
Ratio           : x522.9 plus rapide
Résultats identiques : True


## 3. Broadcasting

Le broadcasting permet d'effectuer des opérations entre des arrays de
formes **différentes** mais **compatibles**.

Règles (simplifiées) :
- On aligne les formes à droite.
- Deux dimensions sont compatibles si elles sont égales, ou si l'une vaut 1.
- La dimension de taille 1 est « étirée » (broadcast) pour correspondre à l'autre.

Exemple : `(4, 4) + (4,)` → le vecteur de taille 4 est étiré sur les 4 lignes.

In [9]:
# Cellule 11 : broadcasting sur l'array 2D existant
# salaires_par_niveau_annee : shape (4, 4)  -> 4 niveaux × 4 années
# moyenne_par_annee : shape (4,)            -> moyenne pour chaque année

moyenne_par_annee = salaires_par_niveau_annee.mean(axis=0)
print("Moyenne par année :", moyenne_par_annee)
print("Shape moyenne_par_annee :", moyenne_par_annee.shape)
print()

# Grâce au broadcasting, on peut centrer chaque salaire par la moyenne de SON année
salaires_centres = salaires_par_niveau_annee - moyenne_par_annee
print("Écarts à la moyenne de l'année (broadcasting) :\n", salaires_centres)
print()

# Vérification : chaque colonne doit sommer à 0 (à l'arrondi près)
print("Somme par colonne :", salaires_centres.sum(axis=0).round(6))

Moyenne par année : [101250. 113750. 125750. 137250.]
Shape moyenne_par_annee : (4,)

Écarts à la moyenne de l'année (broadcasting) :
 [[-56250. -61750. -67750. -75250.]
 [-31250. -35750. -40750. -45250.]
 [  8750.  11250.  14250.  17750.]
 [ 78750.  86250.  94250. 102750.]]

Somme par colonne : [0. 0. 0. 0.]


In [10]:
# Cellule 12 : broadcasting avec un vecteur colonne
# moyenne_par_niveau : shape (4,) -> un salaire moyen par niveau
# Pour la transformer en vecteur colonne (4, 1) compatible avec la matrice (4, 4),
# on utilise reshape(-1, 1).

moyenne_par_niveau = salaires_par_niveau_annee.mean(axis=1)
print("Moyenne par niveau :", moyenne_par_niveau, "| shape :", moyenne_par_niveau.shape)

moyenne_par_niveau_col = moyenne_par_niveau.reshape(-1, 1)
print("En colonne :\n", moyenne_par_niveau_col)
print("Shape :", moyenne_par_niveau_col.shape)
print()

# Centrer chaque ligne par sa propre moyenne
ecarts_ligne = salaires_par_niveau_annee - moyenne_par_niveau_col
print("Écarts à la moyenne du niveau (broadcasting) :\n", ecarts_ligne)
print()

# Vérification : chaque ligne doit sommer à 0
print("Somme par ligne :", ecarts_ligne.sum(axis=1).round(6))

Moyenne par niveau : [ 54250.  81250. 132500. 210000.] | shape : (4,)
En colonne :
 [[ 54250.]
 [ 81250.]
 [132500.]
 [210000.]]
Shape : (4, 1)

Écarts à la moyenne du niveau (broadcasting) :
 [[ -9250.  -2250.   3750.   7750.]
 [-11250.  -3250.   3750.  10750.]
 [-22500.  -7500.   7500.  22500.]
 [-30000. -10000.  10000.  30000.]]

Somme par ligne : [0. 0. 0. 0.]


In [11]:
# Cellule 13 : application métier du broadcasting
print("=== Ajustement uniforme de +7 % sur tous les salaires ===")
salaires_ajustes = salaires_par_niveau_annee * 1.07
print(salaires_ajustes.round(0))
print()

print("=== Comparaison à un seuil de 100 000 USD ===")
depasse_seuil = salaires_par_niveau_annee > 100_000
print(depasse_seuil)
print()

print("=== Nombre de cas > 100k par niveau (ligne) ===")
print("Par niveau :", depasse_seuil.sum(axis=1))
print("Par année  :", depasse_seuil.sum(axis=0))
print()

print("=== Combien de salaires > moyenne de leur année ? ===")
au_dessus_moyenne = salaires_par_niveau_annee > moyenne_par_annee
print(au_dessus_moyenne)
print("Total :", au_dessus_moyenne.sum(), "sur", au_dessus_moyenne.size, "salaires")

=== Ajustement uniforme de +7 % sur tous les salaires ===
[[ 48150.  55640.  62060.  66340.]
 [ 74900.  83460.  90950.  98440.]
 [117700. 133750. 149800. 165850.]
 [192600. 214000. 235400. 256800.]]

=== Comparaison à un seuil de 100 000 USD ===
[[False False False False]
 [False False False False]
 [ True  True  True  True]
 [ True  True  True  True]]

=== Nombre de cas > 100k par niveau (ligne) ===
Par niveau : [0 0 4 4]
Par année  : [2 2 2 2]

=== Combien de salaires > moyenne de leur année ? ===
[[False False False False]
 [False False False False]
 [ True  True  True  True]
 [ True  True  True  True]]
Total : 8 sur 16 salaires


## 4. Statistiques descriptives sur les salaires réels

On charge la colonne `salary_in_usd` du dataset réel (151 445 lignes)
dans un array NumPy, puis on calcule les statistiques fondamentales.
Les valeurs seront comparées à celles obtenues avec Pandas à l'étape 10.

In [12]:
# Cellule 14 : chargement des salaires réels
import csv

salaires_reels = []
with open(CSV_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        salaires_reels.append(int(row["salary_in_usd"]))

salaires_reels = np.array(salaires_reels, dtype=np.int64)

print("Shape :", salaires_reels.shape)
print("dtype :", salaires_reels.dtype)
print("Taille (octets) :", salaires_reels.nbytes)
print("5 premiers :", salaires_reels[:5])
print("5 derniers :", salaires_reels[-5:])

Shape : (151445,)
dtype : int64
Taille (octets) : 1211560
5 premiers : [348516 232344 145400  81600 160000]
5 derniers : [412000 151000 105000 100000  94665]


In [13]:
# Cellule 15 : statistiques fondamentales
moyenne = np.mean(salaires_reels)
mediane = np.median(salaires_reels)
minimum = np.min(salaires_reels)
maximum = np.max(salaires_reels)
ecart_type = np.std(salaires_reels, ddof=1)  # ddof=1 = écart-type de l'échantillon

print(f"Moyenne          : {moyenne:,.2f} USD")
print(f"Médiane          : {mediane:,.2f} USD")
print(f"Minimum          : {minimum:,} USD")
print(f"Maximum          : {maximum:,} USD")
print(f"Écart-type (std) : {ecart_type:,.2f} USD")
print()
print(f"Étendue          : {maximum - minimum:,} USD")
print(f"Coefficient de variation : {ecart_type / moyenne:.2%}")

Moyenne          : 157,527.46 USD
Médiane          : 146,100.00 USD
Minimum          : 15,000 USD
Maximum          : 800,000 USD
Écart-type (std) : 74,150.77 USD

Étendue          : 785,000 USD
Coefficient de variation : 47.07%


In [14]:
# Cellule 16 : quantiles et asymétrie
quantiles = np.percentile(salaires_reels, [1, 5, 25, 50, 75, 95, 99])
labels = ["p1", "p5", "p25 (Q1)", "p50 (médiane)", "p75 (Q3)", "p95", "p99"]
for label, q in zip(labels, quantiles):
    print(f"{label:15s} : {q:,.0f} USD")

print()

# Asymétrie et aplatissement (calcul manuel via NumPy)
mean = np.mean(salaires_reels)
std = np.std(salaires_reels, ddof=0)
skewness = np.mean(((salaires_reels - mean) / std) ** 3)
kurtosis = np.mean(((salaires_reels - mean) / std) ** 4) - 3

print(f"Asymétrie (skewness) : {skewness:.2f}")
print(f"Aplatissement (kurtosis) : {kurtosis:.2f}")

p1              : 36,842 USD
p5              : 60,000 USD
p25 (Q1)        : 105,800 USD
p50 (médiane)   : 146,100 USD
p75 (Q3)        : 198,000 USD
p95             : 290,180 USD
p99             : 385,000 USD

Asymétrie (skewness) : 1.35
Aplatissement (kurtosis) : 4.69


In [15]:
# Cellule 17 : contrôle de cohérence NumPy vs Pandas
import pandas as pd

df_check = pd.read_csv(CSV_PATH, encoding="utf-8", usecols=["salary_in_usd"])

comparaison = pd.DataFrame({
    "NumPy": [
        moyenne,
        mediane,
        minimum,
        maximum,
        ecart_type,
    ],
    "Pandas": [
        df_check["salary_in_usd"].mean(),
        df_check["salary_in_usd"].median(),
        df_check["salary_in_usd"].min(),
        df_check["salary_in_usd"].max(),
        df_check["salary_in_usd"].std(),
    ],
}, index=["moyenne", "médiane", "min", "max", "écart-type"])

comparaison["différence"] = comparaison["NumPy"] - comparaison["Pandas"]
print(comparaison.round(4))
print()
print("Toutes les différences < 0.01 :", (comparaison["différence"].abs() < 0.01).all())

                  NumPy       Pandas  différence
moyenne     157527.4584  157527.4584         0.0
médiane     146100.0000  146100.0000         0.0
min          15000.0000   15000.0000         0.0
max         800000.0000  800000.0000         0.0
écart-type   74150.7724   74150.7724         0.0

Toutes les différences < 0.01 : True


## 5. Interprétation des statistiques sur les salaires réels

### 5.1 Chiffres clés (sur 151 445 salaires)

| Indicateur | Valeur |
|------------|--------|
| Moyenne | 157 527,46 USD |
| Médiane | 146 100,00 USD |
| Écart-type | 74 150,77 USD |
| Coefficient de variation | 47,07 % |
| Minimum | 15 000 USD |
| Maximum | 800 000 USD |
| 1er percentile (p1) | 36 842 USD |
| 25e percentile (Q1) | 105 800 USD |
| 75e percentile (Q3) | 198 000 USD |
| 99e percentile (p99) | 385 000 USD |
| Asymétrie (skewness) | 1,35 |
| Kurtosis | 4,69 |

### 5.2 Interprétation

**1. La moyenne n'est pas un bon indicateur central.**
La moyenne (157 527 USD) dépasse la médiane (146 100 USD) de **11 427 USD**.
Cela signifie qu'une minorité de salaires très élevés tire la moyenne vers le haut.
**Conséquence :** on privilégiera la **médiane** comme indicateur central dans les
analyses descriptives, et on mentionnera systématiquement les deux valeurs.

**2. La distribution est asymétrique à droite.**
L'asymétrie (skewness) de **1,35** confirme cette observation : la queue droite
est plus longue que la queue gauche. C'est la forme attendue pour des salaires
(quelques très hauts revenus, beaucoup de salaires moyens).
**Conséquence pour la modélisation :** une transformation logarithmique
(`np.log1p`) de `salary_in_usd` sera **testée à l'étape 32** pour réduire
l'asymétrie. Elle ne sera retenue que si elle améliore les métriques de manière
mesurable, et les métriques finales seront toujours rapportées en USD.

**3. La dispersion est forte.**
Le coefficient de variation de **47,07 %** indique que l'écart-type représente
presque la moitié de la moyenne. Les salaires sont donc **très hétérogènes**,
ce qui justifie pleinement l'usage de variables explicatives (niveau
d'expérience, métier, pays, télétravail) pour tenter de réduire cette variance.

**4. Les valeurs extrêmes sont légitimes.**
Le minimum (15 000 USD) et le maximum (800 000 USD) ne sont pas des erreurs de
saisie : ils correspondent à des réalités du marché (postes juniors dans des
pays à faible coût de la vie, postes de direction dans de grandes entreprises).
La kurtosis de **4,69** (queues épaisses) confirme que les valeurs extrêmes
sont plus fréquentes qu'une loi normale ne le prédirait.
**Conséquence :** on ne supprime **aucune** valeur extrême. On les documente et
on les conserve.

**5. L'écart interdécile est important.**
Le rapport entre le 99e percentile (385 000 USD) et le 1er percentile
(36 842 USD) est d'environ **10,4×**. Autrement dit, les 1 % les mieux payés
gagnent en moyenne 10 fois plus que les 1 % les moins payés. Ce constat
justifie une analyse par sous-groupes (pays, métiers, expérience) à la phase 4.

**6. 50 % des salaires sont concentrés dans un intervalle étroit.**
L'intervalle interquartile (Q1 → Q3) est de **105 800 → 198 000 USD**, soit
**50 % des salaires dans une fourchette de 92 200 USD**. C'est une information
cruciale : le « cœur du marché » est relativement resserré autour de la médiane.

### 5.3 Décisions pour la suite

- **Cible :** conserver `salary_in_usd` en USD, mais tester `log1p` à l'étape 32.
- **Valeurs extrêmes :** conserver toutes les lignes, aucune suppression.
- **Indicateur central :** privilégier la médiane dans les analyses descriptives.
- **Analyses par sous-groupes :** indispensables pour expliquer la dispersion.
- **Contrôle de cohérence :** NumPy et Pandas donnent des résultats identiques
  (différence = 0,0) — la lecture des données est fiable.